# Notebook 2c: Toxic-BERT Fine-Tuning

## Purpose
This notebook fine-tunes the toxic-BERT model on expert-annotated data to improve performance on the 4-level toxicity classification task:
1. Load and prepare expert-annotated validation dataset
2. Split into training and validation sets (80/20 stratified)
3. Load pre-trained toxic-BERT and modify for 4-class output
4. Fine-tune model with class weights to handle imbalanced data
5. Evaluate fine-tuned model performance
6. Save fine-tuned model and results

## Overview
- **Input**: validation_1043_comments.csv (from Notebook 1)
- **Output**: Fine-tuned model, evaluation metrics, validation predictions
- **Model**: unitary/toxic-bert (fine-tuned for 4-class classification)
- **Training**: CPU-optimized (batch size 4-8, expect 30-60 min per epoch)

In [ ]:
# Install required packages if needed
# pip install transformers torch scikit-learn

In [ ]:
# Basic imports
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn




In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# CHANGE THIS PATH if your file is in a subfolder
csv_path = "/content/drive/MyDrive/Mining/validation_1043_comments.csv"

df = pd.read_csv(csv_path)
df.head()

,article_id,comment_counter,title,globe_url,url,comment_text,is_constructive,is_constructive:confidence,toxicity_level,toxicity_level:confidence,did_you_read_the_article,did_you_read_the_article:confidence,annotator_comments,expert_is_constructive,expert_toxicity_level,expert_comments,Labeled
0,20144737,source1_20144737_8,Enough is enough: Time to address epidemic of ...,http://www.theglobeandmail.com/opinion/put-nat...,http://www.sfu.ca/content/dam/sfu/discourse-la...,All of this energy should be directed at the p...,yes,0.7861,4,0.7758\n0.2242,1,1.0,\nThis does promote further discussion althoug...,no,4.0,"This is really insulting, and I don't see any ...",Yes
1,32803341,source1_32803341_108,"Thank you, Hillary. Now women know retreat is ...",http://www.theglobeandmail.com/opinion/thank-y...,http://www.sfu.ca/content/dam/sfu/discourse-la...,Another load of tosh from a GTA Liberal,no,1.0000,3,0.3628\n0.2748,1,1.0,\n\n\n\n\n\n\n\n\n\n,no,4.0,NaN,Yes
2,20144737,source1_20144737_1_0,Enough is enough: Time to address epidemic of ...,http://www.theglobeandmail.com/opinion/put-nat...,http://www.sfu.ca/content/dam/sfu/discourse-la...,"Unfortunately, this child was taken from her h...",yes,1.0000,2,0.674\n0.326,1,1.0,\n\n,no,4.0,This person is speaking as if she has the insi...,No
3,20144737,source1_20144737_2,Enough is enough: Time to address epidemic of ...,http://www.theglobeandmail.com/opinion/put-nat...,http://www.sfu.ca/content/dam/sfu/discourse-la...,"While the situation is tragic, I object to the...",yes,1.0000,4,1,1,1.0,\n\n,no,4.0,"Demeaning, causes embarrassment and disrespect...",Yes
4,20144737,source1_20144737_3_0,Enough is enough: Time to address epidemic of ...,http://www.theglobeandmail.com/opinion/put-nat...,http://www.sfu.ca/content/dam/sfu/discourse-la...,I agree with her that the problems stem from c...,yes,0.6645,3,1,1,1.0,\n\n,no,4.0,It's insulting. This comment paints all surviv...,Yes


## Section 1: Data Loading and Preparation

In [ ]:
df = df[
    df["comment_text"].notna() &
    df["toxicity_level"].notna()
].copy()

df["toxicity_level"] = df["toxicity_level"].astype(int)

print("Total samples:", len(df))
print(df["toxicity_level"].value_counts().sort_index())


Total samples: 1043
toxicity_level
1    752
2    217
3     57
4     17
Name: count, dtype: int64


## Section 2: Train/Validation Split

In [ ]:
# Split (keep exactly this logic)
X = df["comment_text"].values
y = df["toxicity_level"].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# ---- Class weights (for weighted loss) ----
train_labels = y_train - 1  # convert 1–4 → 0–3

counts = np.bincount(train_labels, minlength=4)
weights = 1.0 / counts
weights = weights / weights.sum() * 4  # normalize (optional but fine)

class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

print("Train class counts:", counts)
print("Class weights:", class_weights)

print("Train:", len(X_train), " Val:", len(X_val))
print("Train dist:", pd.Series(y_train).value_counts().sort_index().to_dict())
print("Val dist:", pd.Series(y_val).value_counts().sort_index().to_dict())


Train class counts: [601 173  46  14]
Class weights: tensor([0.0662, 0.2298, 0.8643, 2.8398], device='cuda:0')
Train: 834  Val: 209
Train dist: {1: 601, 2: 173, 3: 46, 4: 14}
Val dist: {1: 151, 2: 44, 3: 11, 4: 3}


## Section 3: Model Setup - Load and Modify Classification Head

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "unitary/toxic-bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Replace the classifier to output 4 classes
model.config.num_labels = 4
hidden_size = model.config.hidden_size
model.classifier = nn.Linear(hidden_size, 4)

model.to(device)
print("Loaded and moved model to:", device)


Loaded and moved model to: cuda


## Section 5: Dataset and DataLoader Setup

In [ ]:
class ToxicityDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(int(self.labels[idx]) - 1, dtype=torch.long)  # 1-4 -> 0-3
        }


# DataLoaders

In [ ]:
from torch.utils.data import WeightedRandomSampler

batch_size = 16
max_length = 256

train_ds = ToxicityDataset(X_train, y_train, tokenizer, max_length=max_length)
val_ds   = ToxicityDataset(X_val, y_val, tokenizer, max_length=max_length)

# ---- Oversampling (TRAIN ONLY) ----
train_labels = y_train - 1  # 0–3
class_counts = np.bincount(train_labels, minlength=4)
class_weights_np = 1.0 / class_counts
sample_weights = class_weights_np[train_labels]


train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False
)


print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))


Train batches: 53
Val batches: 14


## Section 6: Training Configuration

In [ ]:
# Training hyperparameters and optimization setup

learning_rate = 1e-5
num_epochs = 6
weight_decay = 0.01
early_stopping_patience = 1

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
)

criterion = nn.CrossEntropyLoss(weight=class_weights)


## Section 7: Training Loop

In [ ]:
# Training and validation loop with early stopping

train_losses, val_losses = [], []
train_f1s, val_f1s = [], []

best_val_f1 = -1
patience_counter = 0
best_model_state = None

for epoch in range(num_epochs):

    model.train()
    total_loss = 0
    all_preds, all_true = [], []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} Train"):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_true.extend(labels.cpu().numpy())

    avg_train_loss = total_loss / len(train_loader)
    train_f1 = f1_score(all_true, all_preds, average="macro")

    train_losses.append(avg_train_loss)
    train_f1s.append(train_f1)

    model.eval()
    total_loss = 0
    all_preds, all_true = [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} Val"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            loss = criterion(logits, labels)

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_true.extend(labels.cpu().numpy())

    avg_val_loss = total_loss / len(val_loader)
    val_f1 = f1_score(all_true, all_preds, average="macro")

    val_losses.append(avg_val_loss)
    val_f1s.append(val_f1)

    print(f"\nEpoch {epoch+1}")
    print(f"Train loss: {avg_train_loss:.4f} | Train macro-F1: {train_f1:.4f}")
    print(f"Val   loss: {avg_val_loss:.4f} | Val   macro-F1: {val_f1:.4f}")
    print("-" * 50)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = model.state_dict()
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= early_stopping_patience:
        break

model.load_state_dict(best_model_state)


Epoch 1 Val: 100%|██████████| 14/14 [00:02<00:00,  4.78it/s]



Epoch 1
Train loss: 1.3596 | Train macro-F1: 0.2175
Val   loss: 1.2891 | Val   macro-F1: 0.2779
--------------------------------------------------


Epoch 2 Val: 100%|██████████| 14/14 [00:03<00:00,  4.61it/s]



Epoch 2
Train loss: 1.2763 | Train macro-F1: 0.3384
Val   loss: 1.2423 | Val   macro-F1: 0.2879
--------------------------------------------------


Epoch 3 Val: 100%|██████████| 14/14 [00:02<00:00,  4.69it/s]


...

Epoch 4 Val: 100%|██████████| 14/14 [00:02<00:00,  4.67it/s]



Epoch 4
Train loss: 1.0298 | Train macro-F1: 0.5500
Val   loss: 1.1964 | Val   macro-F1: 0.4094
--------------------------------------------------


Epoch 5 Val: 100%|██████████| 14/14 [00:02<00:00,  4.70it/s]


Epoch 5
Train loss: 0.8596 | Train macro-F1: 0.7036
Val   loss: 1.2242 | Val   macro-F1: 0.3974
--------------------------------------------------


<All keys matched successfully>

## Section 8: Model Evaluation

In [ ]:
model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        preds = torch.argmax(logits, dim=1)

        all_preds.extend((preds.cpu().numpy() + 1))   # back to 1–4
        all_true.extend((labels.cpu().numpy() + 1))   # back to 1–4

print("Accuracy:", accuracy_score(all_true, all_preds))
print(classification_report(all_true, all_preds, labels=[1,2,3,4], digits=4))


Accuracy: 0.6028708133971292
              precision    recall  f1-score   support

           1     0.7794    0.7020    0.7387       151
           2     0.2833    0.3864    0.3269        44
           3     0.2000    0.1818    0.1905        11
           4     0.3333    0.3333    0.3333         3

    accuracy                         0.6029       209
   macro avg     0.3990    0.4009    0.3974       209
weighted avg     0.6381    0.6029    0.6173       209



## Section 9: Save Results

In [ ]:
save_dir = "/content/drive/MyDrive/toxic_bert_finetuned_4class"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print("Saved to:", save_dir)


Saved to: /content/drive/MyDrive/toxic_bert_finetuned_4class
